In [6]:
import asyncio
import websockets
import wave
import json

async def send_audio(file_path):
    uri = "ws://localhost:8000/ws/predict"  # Make sure this is your actual WebSocket server address
    async with websockets.connect(uri) as websocket:
        print("🔌 Connected to WebSocket")

        with wave.open(file_path, 'rb') as wav_file:
            frame_rate = wav_file.getframerate()
            samp_width = wav_file.getsampwidth()
            chunk_duration = 1  # seconds
            chunk_size = frame_rate * samp_width * wav_file.getnchannels() * chunk_duration

            while True:
                chunk = wav_file.readframes(chunk_size // samp_width)
                if not chunk:
                    break
                await websocket.send(chunk)

                try:
                    response = await asyncio.wait_for(websocket.recv(), timeout=2)
                    data = json.loads(response)
                    print("🧠 Prediction:", data.get("predicted_class"))
                except asyncio.TimeoutError:
                    print("⚠️ Timeout waiting for response")

# Run this line inside your Jupyter cell:
await send_audio("data/urbansound8k_one_clip_per_class_with_silence (2).wav")


🔌 Connected to WebSocket
⚠️ Timeout waiting for response
🧠 Prediction: No Anomaly
🧠 Prediction: dog_bark
🧠 Prediction: children_playing
🧠 Prediction: car_horn
🧠 Prediction: children_playing
🧠 Prediction: children_playing
🧠 Prediction: No Anomaly
🧠 Prediction: dog_bark
🧠 Prediction: dog_bark
🧠 Prediction: dog_bark
🧠 Prediction: dog_bark
🧠 Prediction: dog_bark
🧠 Prediction: dog_bark
🧠 Prediction: dog_bark
🧠 Prediction: dog_bark
🧠 Prediction: dog_bark
🧠 Prediction: No Anomaly
🧠 Prediction: No Anomaly
🧠 Prediction: dog_bark
🧠 Prediction: No Anomaly


In [23]:
import asyncio
import websockets
import wave
import json
import os

async def send_audio(file_path):
    # Validate file existence
    if not os.path.exists(file_path):
        print(f"❌ Error: File {file_path} does not exist")
        return

    uri = "ws://localhost:8000/ws/predict"  # WebSocket server address
    try:
        async with websockets.connect(uri) as websocket:
            print("🔌 Connected to WebSocket")

            with wave.open(file_path, 'rb') as wav_file:
                # Get WAV file parameters
                frame_rate = wav_file.getframerate()
                samp_width = wav_file.getsampwidth()
                n_channels = wav_file.getnchannels()
                n_frames = wav_file.getnframes()

                print(f"🎵 Audio Info: Sample Rate={frame_rate} Hz, Sample Width={samp_width} bytes, "
                      f"Channels={n_channels}, Total Frames={n_frames}")

                # Server expects 3-second chunks, 48000 Hz, stereo, 16-bit
                expected_sample_rate = 48000
                expected_channels = 2
                expected_samp_width = 2
                chunk_duration = 3  # Match server's chunk duration
                frame_size = expected_channels * expected_samp_width
                chunk_size = int(expected_sample_rate * frame_size * chunk_duration)

                # Validate audio format
                if frame_rate != expected_sample_rate or n_channels != expected_channels or samp_width != expected_samp_width:
                    print(f"⚠️ Warning: Audio format mismatch. Expected {expected_sample_rate} Hz, {expected_channels} channels, "
                          f"{expected_samp_width*8}-bit. Got {frame_rate} Hz, {n_channels} channels, {samp_width*8}-bit.")
                    # Optionally, resample/convert audio here using pydub if needed

                while True:
                    chunk = wav_file.readframes(chunk_size // samp_width)
                    if not chunk:
                        print("🏁 End of audio file")
                        break

                    print(f"📤 Sending chunk of size: {len(chunk)} bytes")
                    await websocket.send(chunk)

                    try:
                        # Increase timeout to account for server processing
                        response = await asyncio.wait_for(websocket.recv(), timeout=15)
                        try:
                            data = json.loads(response)
                            print(f"🧠 Prediction: {data.get('predicted_class')}")
                        except json.JSONDecodeError:
                            print(f"❌ Invalid JSON response: {response}")
                    except asyncio.TimeoutError:
                        print("⚠️ Timeout waiting for server response")
                    except websockets.exceptions.ConnectionClosed:
                        print("❌ WebSocket connection closed unexpectedly")
                        break

    except websockets.exceptions.InvalidURI:
        print(f"❌ Invalid WebSocket URI: {uri}")
    except websockets.exceptions.InvalidHandshake:
        print("❌ WebSocket handshake failed")
    except Exception as e:
        print(f"❌ Error: {str(e)}")

# Example usage in a Jupyter cell:
# await send_audio("data/urbansound8k_one_clip_per_class_with_silence (2).wav")

In [24]:
await send_audio("data/urbansound8k_one_clip_per_class_with_silence (2).wav")

🔌 Connected to WebSocket
🎵 Audio Info: Sample Rate=48000 Hz, Sample Width=4 bytes, Channels=2, Total Frames=1995617
⚠️ Warning: Audio format mismatch. Expected 48000 Hz, 2 channels, 16-bit. Got 48000 Hz, 2 channels, 32-bit.
📤 Sending chunk of size: 1152000 bytes
⚠️ Timeout waiting for server response
📤 Sending chunk of size: 1152000 bytes
🧠 Prediction: dog_bark
📤 Sending chunk of size: 1152000 bytes
🧠 Prediction: children_playing
📤 Sending chunk of size: 1152000 bytes
🧠 Prediction: children_playing
📤 Sending chunk of size: 1152000 bytes
🧠 Prediction: children_playing
📤 Sending chunk of size: 1152000 bytes
🧠 Prediction: drilling
📤 Sending chunk of size: 1152000 bytes
🧠 Prediction: dog_bark
📤 Sending chunk of size: 1152000 bytes
🧠 Prediction: dog_bark
📤 Sending chunk of size: 1152000 bytes
🧠 Prediction: No Anomaly
📤 Sending chunk of size: 1152000 bytes
🧠 Prediction: No Anomaly
📤 Sending chunk of size: 1152000 bytes
🧠 Prediction: No Anomaly
📤 Sending chunk of size: 1152000 bytes
🧠 Predict

In [2]:
from main import continuous_predict

In [11]:
import requests

# Replace with your actual API endpoint
url = "http://localhost:8000/continuous_predict"
audio_file_path = "data/urbansound8k_one_clip_per_class_with_silence.wav"

with open(audio_file_path, "rb") as f:
    files = {"audio": (audio_file_path, f, "audio/wav")}
    response = requests.post(url, files=files)

if response.ok:
    
    print("API Response:", response.json())
else:
    print("Error:", response.status_code, response.text)

API Response: {'predicted_class': [[0.0, 2.0, 'dog_bark', 0.9957864880561829], [1.0, 3.0, 'children_playing', 0.995547890663147], [2.0, 4.0, 'children_playing', 0.9921167492866516], [3.0, 5.0, 'children_playing', 0.99986732006073], [4.0, 6.0, 'drilling', 0.9952110648155212], [5.0, 7.0, 'dog_bark', 0.999995231628418], [6.0, 8.0, 'dog_bark', 0.9957804679870605], [7.0, 9.0, 'No Anomaly', 0.6280861496925354], [8.0, 10.0, 'No Anomaly', 0.645057201385498], [9.0, 11.0, 'No Anomaly', 0.4351494312286377], [10.0, 12.0, 'jackhammer', 0.9869378805160522], [11.0, 13.0, 'No Anomaly', 0.7511363625526428], [12.0, 14.0, 'No Anomaly', 0.5286034941673279], [13.0, 15.0, 'street_music', 0.9040104150772095], [14.0, 16.0, 'No Anomaly', 0.7484550476074219], [15.0, 17.0, 'street_music', 0.8706051707267761], [16.0, 18.0, 'No Anomaly', 0.3359003961086273], [17.0, 19.0, 'dog_bark', 0.9589837789535522], [18.0, 20.0, 'No Anomaly', 0.4105808138847351], [19.0, 21.0, 'dog_bark', 0.9892774820327759], [20.0, 22.0, 'dog_

In [4]:
predictions = continuous_predict("data/urbansound8k_one_clip_per_class_with_silence.wav", window_size=2.0, hop_size=1.0, threshold=0.8)
for start, end, pred, conf in predictions:
    print(f"Time [{start:.2f}s - {end:.2f}s]: {pred} (confidence: {conf:.3f})")

🔊 Processing audio stream from data/urbansound8k_one_clip_per_class_with_silence.wav
Window [0.00s - 2.00s]: dog_bark (confidence: 0.996)
Window [1.00s - 3.00s]: children_playing (confidence: 0.996)
Window [2.00s - 4.00s]: children_playing (confidence: 0.992)
Window [3.00s - 5.00s]: children_playing (confidence: 1.000)
Window [4.00s - 6.00s]: drilling (confidence: 0.995)
Window [5.00s - 7.00s]: dog_bark (confidence: 1.000)
Window [6.00s - 8.00s]: dog_bark (confidence: 0.996)
Window [7.00s - 9.00s]: No Anomaly (confidence: 0.628)
Window [8.00s - 10.00s]: No Anomaly (confidence: 0.645)
Window [9.00s - 11.00s]: No Anomaly (confidence: 0.435)
Window [10.00s - 12.00s]: jackhammer (confidence: 0.987)
Window [11.00s - 13.00s]: No Anomaly (confidence: 0.751)
Window [12.00s - 14.00s]: No Anomaly (confidence: 0.529)
Window [13.00s - 15.00s]: street_music (confidence: 0.904)
Window [14.00s - 16.00s]: No Anomaly (confidence: 0.748)
Window [15.00s - 17.00s]: street_music (confidence: 0.871)
Window 

In [7]:
import wave
import shutil
from tempfile import NamedTemporaryFile

def save_wav_with_new_name(input_wav_path, new_filename="copied.wav"):
    with wave.open(input_wav_path, 'rb') as original:
        params = original.getparams()
        frames = original.readframes(params.nframes)
        bytes_per_sample = original.getsampwidth()
        sample_rate = params.framerate

    with NamedTemporaryFile(delete=False, suffix=".wav") as temp_file:
        temp_file_path = temp_file.name
        with wave.open(temp_file, 'wb') as wav_file:
            wav_file.setnchannels(params.nchannels)
            wav_file.setsampwidth(bytes_per_sample)
            wav_file.setframerate(sample_rate)
            wav_file.writeframes(frames)

    # Rename temp file to desired output name
    shutil.move(temp_file_path, new_filename)
    print(f"Saved new file as: {new_filename}")


In [8]:
save_wav_with_new_name("data/7062-6-0-0.wav", new_filename="new_clip.wav")


Saved new file as: new_clip.wav


In [1]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

In [1]:
import os
import zipfile
import urllib.request

# Define dataset URL and extraction paths
dataset_url = "https://zenodo.org/record/1203745/files/UrbanSound8K.tar.gz"
dataset_path = "./UrbanSound8K"

# Download and extract the dataset
if not os.path.exists(dataset_path):
    os.makedirs(dataset_path)
    zip_path = os.path.join(dataset_path, "UrbanSound8K.tar.gz")
    urllib.request.urlretrieve(dataset_url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(dataset_path)


KeyboardInterrupt: 

In [7]:
import requests

# Replace with your actual API endpoint
url = "http://localhost:8000/history"

response = requests.get(url)
if response.ok:
    print("History:", response.json())
else:
    print("Error:", response.status_code, response.text)

History: {'history': [{'predicted_class': 'No Anomaly', 'confidence': 0.0, 'timestamp': '2025-05-31 02:41:23.933456'}, {'predicted_class': 'No Anomaly', 'confidence': 0.0, 'timestamp': '2025-05-31 02:41:20.872926'}, {'predicted_class': 'street_music', 'confidence': 0.970024049282074, 'timestamp': '2025-05-31 02:41:17.773203'}, {'predicted_class': 'No Anomaly', 'confidence': 0.0, 'timestamp': '2025-05-31 02:41:14.720089'}, {'predicted_class': 'No Anomaly', 'confidence': 0.0, 'timestamp': '2025-05-31 02:41:11.625955'}, {'predicted_class': 'children_playing', 'confidence': 0.8305137157440186, 'timestamp': '2025-05-31 02:41:05.183487'}, {'predicted_class': 'street_music', 'confidence': 0.9999998807907104, 'timestamp': '2025-05-31 02:41:02.118443'}, {'predicted_class': 'street_music', 'confidence': 0.9974820017814636, 'timestamp': '2025-05-31 02:40:59.033369'}, {'predicted_class': 'street_music', 'confidence': 0.9959672689437866, 'timestamp': '2025-05-31 02:40:55.956178'}, {'predicted_class